#### Task 1: Build a Complete ETL Pipeline
Build an ETL pipeline that processes raw e-commerce order data into a clean analytical dataset.

**Step 1** — Generate raw data: Create a list of 200 raw order records as dictionaries with these fields: order_id, customer_id, product_name, quantity, unit_price, order_date, shipping_country. Intentionally include at least 15 problematic records across these categories:

- 3 records with missing product_name (None or empty string)
- 3 records with negative quantity or unit_price
- 3 records with malformed order_date (e.g., "not-a-date", "2025-13-45")
- 3 records with duplicate order_id values
- 3 records where quantity or unit_price is a string instead of a number
  
**Step 2** — Extract with validation: Write an extract(raw_records) function that:

- Parses each record and validates required fields exist
- Validates that quantity and unit_price are positive numbers
- Validates that order_date is a parseable date
- Returns two lists: valid_records and rejected_records
- Each rejected record should include the reason for rejection
- Print a summary: count of valid, count of rejected by reason

**Step 3** — Transform: Write a transform(valid_records) function that:

- Converts to a pandas DataFrame
- Computes total_amount = quantity * unit_price
- Extracts order_month and order_day_of_week from the date
- Standardizes shipping_country to title case
- Removes duplicate order_id values (keep the first)
- Adds an amount_category column: "small" (<$25), "medium" ($25-$100), "large" (>$100)
- Returns the cleaned DataFrame
  
**Step 4**— Load: Write a load(df, path) function that saves the DataFrame to a Parquet file. Then read it back and verify the row count and dtypes match.

**Step 5:** Run the full pipeline: extract → transform → load. Print a summary showing records at each stage (raw → valid → transformed → loaded).

In [1]:
#step 1 
import random
import pandas as pd
from datetime import datetime, timedelta

products = ["Laptop", "Mouse", "Keyboard", "Monitor", "Headphones"]
countries = ["usa", "germany", "france", "canada", "japan"]

raw_records = []

base_date = datetime(2025, 1, 1)

for i in range(185):
    record = {
        "order_id": i + 1,
        "customer_id": random.randint(1000, 2000),
        "product_name": random.choice(products),
        "quantity": random.randint(1, 5),
        "unit_price": round(random.uniform(10, 200), 2),
        "order_date": (base_date + timedelta(days=random.randint(0, 200))).strftime("%Y-%m-%d"),
        "shipping_country": random.choice(countries)
    }
    
    raw_records.append(record)

# 3 missing product_name
for i in range(3):
    raw_records.append({
        "order_id": 200 + i,
        "customer_id": 1500,
        "product_name": None,
        "quantity": 2,
        "unit_price": 50,
        "order_date": "2025-05-10",
        "shipping_country": "usa"
    })

# 3 negative quantity or price
for i in range(3):
    raw_records.append({
        "order_id": 210 + i,
        "customer_id": 1600,
        "product_name": "Laptop",
        "quantity": -2,
        "unit_price": 100,
        "order_date": "2025-04-10",
        "shipping_country": "canada"
    })

# 3 malformed dates
bad_dates = ["not-a-date", "2025-13-45", "2025/99/10"]

for i in range(3):
    raw_records.append({
        "order_id": 220 + i,
        "customer_id": 1700,
        "product_name": "Mouse",
        "quantity": 1,
        "unit_price": 20,
        "order_date": bad_dates[i],
        "shipping_country": "germany"
    })

# 3 duplicate order_id
for i in range(3):
    raw_records.append({
        "order_id": 1,
        "customer_id": 1800,
        "product_name": "Monitor",
        "quantity": 1,
        "unit_price": 150,
        "order_date": "2025-06-01",
        "shipping_country": "france"
    })

# 3 string numbers
for i in range(3):
    raw_records.append({
        "order_id": 230 + i,
        "customer_id": 1900,
        "product_name": "Keyboard",
        "quantity": "3",
        "unit_price": "40",
        "order_date": "2025-07-01",
        "shipping_country": "japan"
    })

print("Total raw records:", len(raw_records))

Total raw records: 200


In [2]:
#step2
def extract(raw_records):

    valid_records = []
    rejected_records = []
    reject_reasons = {}

    for r in raw_records:
        reason = None

        if not r.get("product_name"):
            reason = "missing_product_name"

        elif not isinstance(r["quantity"], (int, float)) or not isinstance(r["unit_price"], (int, float)):
            reason = "non_numeric_quantity_or_price"

        elif r["quantity"] <= 0 or r["unit_price"] <= 0:
            reason = "negative_quantity_or_price"


        
        else:
            try:
                datetime.strptime(r["order_date"], "%Y-%m-%d")
            except:
                reason = "invalid_date"

        if reason:
            rejected_records.append({**r, "reason": reason})
            reject_reasons[reason] = reject_reasons.get(reason, 0) + 1
        else:
            valid_records.append(r)

    print("VALID RECORDS:", len(valid_records))
    print("REJECTED RECORDS:", len(rejected_records))
    print("REJECTION SUMMARY:", reject_reasons)

    return valid_records, rejected_records
valid_records, rejected_records = extract(raw_records)

VALID RECORDS: 188
REJECTED RECORDS: 12
REJECTION SUMMARY: {'missing_product_name': 3, 'negative_quantity_or_price': 3, 'invalid_date': 3, 'non_numeric_quantity_or_price': 3}


In [3]:
#step3
def transform(valid_records):

    df = pd.DataFrame(valid_records)

    df["order_date"] = pd.to_datetime(df["order_date"])

    df["total_amount"] = df["quantity"] * df["unit_price"]

    df["order_month"] = df["order_date"].dt.month

    df["order_day_of_week"] = df["order_date"].dt.day_name()

    df["shipping_country"] = df["shipping_country"].str.title()

    df = df.drop_duplicates(subset="order_id", keep="first")

    def categorize(x):
        if x < 25:
            return "small"
        elif x <= 100:
            return "medium"
        else:
            return "large"

    df["amount_category"] = df["total_amount"].apply(categorize)

    print("Transformed rows:", len(df))

    return df
df_clean = transform(valid_records)

Transformed rows: 185


In [4]:
#step4
def load(df, path):

    df.to_parquet(path)
    df_loaded = pd.read_parquet(path)

    print("Saved rows:", len(df))
    print("Loaded rows:", len(df_loaded))
    print("Dtypes match:", df.dtypes.equals(df_loaded.dtypes))

    return df_loaded
df_loaded = load(df_clean, "orders_clean.parquet")

Saved rows: 185
Loaded rows: 185
Dtypes match: True


In [5]:
#step5
print("RAW RECORDS:", len(raw_records))

valid_records, rejected_records = extract(raw_records)

df_transformed = transform(valid_records)

df_loaded = load(df_transformed, "orders_clean.parquet")

print("\nPIPELINE SUMMARY")
print("Raw:", len(raw_records))
print("Valid:", len(valid_records))
print("Transformed:", len(df_transformed))
print("Loaded:", len(df_loaded))

RAW RECORDS: 200
VALID RECORDS: 188
REJECTED RECORDS: 12
REJECTION SUMMARY: {'missing_product_name': 3, 'negative_quantity_or_price': 3, 'invalid_date': 3, 'non_numeric_quantity_or_price': 3}
Transformed rows: 185
Saved rows: 185
Loaded rows: 185
Dtypes match: True

PIPELINE SUMMARY
Raw: 200
Valid: 188
Transformed: 185
Loaded: 185


#### Task 2: ETL vs ELT Comparison
**Step 1:** Implement an ELT variant of the pipeline from Task 1:

- Extract: do minimal validation (only reject unparseable JSON/records)
- Load: save the raw (mostly unvalidated) data to a Parquet "data lake" file
- Transform: read from the data lake file, apply the same validation and transformation rules as Task 1

**Step 2:** Compare ETL and ELT approaches. Write a markdown cell addressing:

- How many records made it to the destination in each approach?
- At what stage were problems caught in each?
- What are the advantages of each approach?
- When would you choose one over the other?

In [6]:
#step1
def extract_elt(raw_records):

    extracted = []
    rejected = []

    for r in raw_records:
        if isinstance(r, dict):
            extracted.append(r)
        else:
            rejected.append(r)

    print("Extracted records:", len(extracted))
    print("Rejected (invalid format):", len(rejected))

    return extracted
elt_extracted = extract_elt(raw_records)

Extracted records: 200
Rejected (invalid format): 0


In [7]:
def load_raw_data_lake(records, path):

    df_raw = pd.DataFrame(records)

    df_raw = df_raw.astype(str)

    df_raw.to_parquet(path)

    print("Raw data saved to data lake")
    print("Rows saved:", len(df_raw))

    return path
data_lake_path = load_raw_data_lake(elt_extracted, "raw_data_lake.parquet")

Raw data saved to data lake
Rows saved: 200


In [8]:
def transform_from_lake(path):

    df = pd.read_parquet(path)

    df["quantity"] = pd.to_numeric(df["quantity"], errors="coerce")
    df["unit_price"] = pd.to_numeric(df["unit_price"], errors="coerce")

    valid_records = []
    rejected_records = []

    for _, r in df.iterrows():
        reason = None

        if not r["product_name"]:
            reason = "missing_product_name"

        elif not isinstance(r["quantity"], (int, float)) or not isinstance(r["unit_price"], (int, float)):
            reason = "non_numeric_quantity_or_price"

        elif r["quantity"] <= 0 or r["unit_price"] <= 0:
            reason = "negative_quantity_or_price"

        else:
            try:
                pd.to_datetime(r["order_date"])
            except:
                reason = "invalid_date"

        if reason:
            rejected_records.append(reason)
        else:
            valid_records.append(r)

    df_valid = pd.DataFrame(valid_records)
    df_valid["order_date"] = pd.to_datetime(df_valid["order_date"])
    df_valid["total_amount"] = df_valid["quantity"] * df_valid["unit_price"]
    df_valid["order_month"] = df_valid["order_date"].dt.month
    df_valid["order_day_of_week"] = df_valid["order_date"].dt.day_name()
    df_valid["shipping_country"] = df_valid["shipping_country"].str.title()

    df_valid = df_valid.drop_duplicates(subset="order_id", keep="first")

    def category(x):
        if x < 25:
            return "small"
        elif x <= 100:
            return "medium"
        else:
            return "large"

    df_valid["amount_category"] = df_valid["total_amount"].apply(category)

    print("Valid after transform:", len(df_valid))
    print("Rejected during transform:", len(rejected_records))

    return df_valid
elt_final_df = transform_from_lake(data_lake_path)

Valid after transform: 191
Rejected during transform: 6


#step2
#### ETL vs ELT Comparison

**How many records made it to the destination?**

In both approaches, **191 records** reached the final destination dataset, while **6 records were rejected** due to validation errors.

**At what stage were problems caught?**

- In the **ETL pipeline**, problems were detected during the **extract/validation stage before loading** the data.
- In the **ELT pipeline**, raw data was **loaded first**, and issues were detected during the **transform stage**.

**Advantages of each approach**

**ETL**
- Data is cleaned before storage.
- Ensures higher data quality in the destination system.

**ELT**
- Raw data is preserved in the data lake.
- Allows flexible transformations later.

**When to choose each**

- **ETL** is preferred when strict data quality is required before loading into a data warehouse.
- **ELT** is better for large-scale data lakes where transformations can be applied after loading.

#### Task 3: Simulate Modes of Dataflow
Model a simplified analytics system with three components: an order ingestion service, a feature computation service, and a prediction service.

**Step 1** — Data passing through a database: Simulate a shared database using a dictionary. The order service writes new orders to the database. The feature service reads orders and computes features (total orders, average amount, last order date per customer). The prediction service reads the features.

##### Shared database (simulated)
database = {"orders": [], "features": {}}

def order_service_write(order):
    database["orders"].append(order)

def feature_service_compute():
    # Read from database, compute features, write back
    ...

def prediction_service_read(customer_id):
    # Read features from database
    ...
    
**Step 2** — Data passing through services: Refactor so the prediction service requests data from the feature service directly (function call simulating an API request). The feature service requests raw orders from the order service.

**Step 3**— Data passing through a message broker: Implement a simple pubsub broker (a class with publish, subscribe, and get_latest methods). The order service publishes new orders to an "orders" topic. The feature service subscribes and computes features whenever new orders arrive, publishing them to a "features" topic. The prediction service subscribes to "features."

**Step 4:** Run 20 new orders through all three dataflow modes. Verify that the prediction service gets the same features in each case. Write a markdown cell comparing the three modes: coupling, latency characteristics, and what happens when one component goes down.

In [9]:
from datetime import datetime
import random

orders = []

for i in range(20):
    orders.append({
        "customer_id": random.randint(1,3),
        "amount": random.randint(10,100),
        "date": datetime(2026,1,1)
    })

In [22]:
#step1
database = {"orders": [], "features": {}}

def order_service_write(order):
    database["orders"].append(order)


def feature_service_compute():
    features = {}

    for o in database["orders"]:
        cid = o["customer_id"]

        if cid not in features:
            features[cid] = {"count":0,"sum":0,"last":None}

        features[cid]["count"] += 1
        features[cid]["sum"] += o["amount"]
        features[cid]["last"] = o["date"]

    for cid in features:
        f = features[cid]

        database["features"][cid] = {
            "total_orders": f["count"],
            "avg_amount": f["sum"]/f["count"],
            "last_order": f["last"]
        }

def prediction_service_read(cid):
    return database["features"].get(cid)

In [11]:
for o in orders:
    order_service_write(o)
feature_service_compute()

print("STEP1:", prediction_service_read(1))

STEP1: {'total_orders': 6, 'avg_amount': 47.0, 'last_order': datetime.datetime(2026, 1, 1, 0, 0)}


In [12]:
#step2
orders_service = []

def order_add(order):
    orders_service.append(order)

def order_get():
    return orders_service


def feature_get():
    features = {}

    for o in order_get():
        cid = o["customer_id"]
        if cid not in features:
            features[cid] = {"count":0,"sum":0,"last":None}

        features[cid]["count"] += 1
        features[cid]["sum"] += o["amount"]
        features[cid]["last"] = o["date"]

    result = {}

    for cid in features:
        f = features[cid]
        result[cid] = {
            "total_orders": f["count"],
            "avg_amount": f["sum"]/f["count"],
            "last_order": f["last"]
        }

    return result

def prediction(cid):
    return feature_get().get(cid)

In [13]:
for o in orders:
    order_add(o)

print("STEP2:", prediction(1))

STEP2: {'total_orders': 6, 'avg_amount': 47.0, 'last_order': datetime.datetime(2026, 1, 1, 0, 0)}


In [14]:
#step3
class Broker:

    def __init__(self):
        self.topics = {}

    def publish(self,topic,msg):
        if topic not in self.topics:
            self.topics[topic] = []
        self.topics[topic].append(msg)

    def get(self,topic):
        return self.topics.get(topic,[])

broker = Broker()

features = {}

def order_publish(order):
    broker.publish("orders",order)

def feature_process():

    for o in broker.get("orders"):
        cid = o["customer_id"]

        if cid not in features:
            features[cid]={"count":0,"sum":0,"last":None}

        features[cid]["count"]+=1
        features[cid]["sum"]+=o["amount"]
        features[cid]["last"]=o["date"]

    for cid in features:
        f=features[cid]

        broker.publish("features",{
            "customer_id":cid,
            "total_orders":f["count"],
            "avg_amount":f["sum"]/f["count"],
            "last_order":f["last"]
        })

def prediction_broker(cid):
    for f in broker.get("features"):
        if f["customer_id"]==cid:
            return f

In [15]:
for o in orders:
    order_publish(o)

feature_process()

print("STEP3:", prediction_broker(1))

STEP3: {'customer_id': 1, 'total_orders': 6, 'avg_amount': 47.0, 'last_order': datetime.datetime(2026, 1, 1, 0, 0)}


#step4
#### Comparison of Dataflow Modes

**Database Mode**

All services communicate through a shared database.  
The order service writes orders to the database, the feature service reads them and computes features, and the prediction service reads the computed features.

Coupling: Medium (all services depend on the same database)  
Latency: Medium  
Failure: If the database fails, all services stop working.

---

**Service-to-Service Mode**

Services communicate directly via function calls (simulating APIs).  
The prediction service calls the feature service, and the feature service calls the order service.

Coupling: High (services depend on each other directly)  
Latency: Low  
Failure: If one service fails, dependent services cannot work.

---

**Message Broker Mode**

Services communicate through a message broker using publish-subscribe.

Coupling: Low (services are independent)  
Latency: Very low (event-driven processing)  
Failure: If one service goes down, messages can still be processed later.

#### Task 4: Batch Processing vs Stream Processing
**Step 1** — Batch processing: Using the cleaned data from Task 1, write a batch processing function that computes daily aggregate features:

Total orders per day
Total revenue per day
Average order size per day
Number of unique customers per day
Top product per day (by revenue)
Execute it and display the results.

**Step 2** — Stream processing: Implement a StreamProcessor class that processes orders one at a time and maintains running statistics:

Running total of orders and revenue
Windowed average (last 50 orders) of order size
Running count of unique customers
Current most popular product (by count in the last 50 orders)
Process the same data from Task 1 through the stream processor, one record at a time. After every 50 records, print the current streaming statistics.

**Step 3** — Compare final results: After processing all records through both approaches, compare the final totals. They should match for cumulative statistics (total orders, total revenue). Write a markdown cell explaining: Why might windowed streaming statistics differ from daily batch statistics even on the same data? What does each approach tell you that the other does not?

In [16]:
#step1
import pandas as pd
from collections import deque, Counter
df = df_clean
def batch_processing(df):
    daily = df.groupby("order_date").agg(
        total_orders=("order_id", "count"),
        total_revenue=("total_amount", "sum"),
        avg_order_size=("total_amount", "mean"),
        unique_customers=("customer_id", "nunique")
    ).reset_index()

    top_products = (
        df.groupby(["order_date", "product_name"])["total_amount"]
        .sum()
        .reset_index()
        .sort_values(["order_date", "total_amount"], ascending=[True, False])
        .drop_duplicates("order_date")
        .rename(columns={"product_name": "top_product"})
    )

    result = daily.merge(top_products[["order_date", "top_product"]], on="order_date")
    return result

batch_results = batch_processing(df)
print("=== Batch Processing Results ===")
print(batch_results.head())

=== Batch Processing Results ===
  order_date  total_orders  total_revenue  avg_order_size  unique_customers  \
0 2025-01-01             2         730.45         365.225                 2   
1 2025-01-03             1         208.84         208.840                 1   
2 2025-01-04             2         617.20         308.600                 2   
3 2025-01-05             1         159.33         159.330                 1   
4 2025-01-06             2         284.92         142.460                 2   

  top_product  
0  Headphones  
1    Keyboard  
2     Monitor  
3       Mouse  
4      Laptop  


In [17]:
#step2
class StreamProcessor:
    def __init__(self, window_size=50):
        self.total_orders = 0
        self.total_revenue = 0
        self.customers = set()
        self.window = deque(maxlen=window_size)
        self.products = deque(maxlen=window_size)

    def process(self, order):
        amount = order["total_amount"]
        self.total_orders += 1
        self.total_revenue += amount
        self.customers.add(order["customer_id"])
        self.window.append(amount)
        self.products.append(order["product_name"])

    def stats(self):
        window_avg = sum(self.window) / len(self.window) if self.window else 0
        popular = Counter(self.products).most_common(1)[0][0] if self.products else None
        return {
            "total_orders": self.total_orders,
            "total_revenue": self.total_revenue,
            "unique_customers": len(self.customers),
            "window_avg_order_size": window_avg,
            "popular_product_last50": popular
        }

processor = StreamProcessor(window_size=50)
for i, row in df.iterrows():
    processor.process(row)
    if (i + 1) % 50 == 0:
        print(f"\n=== Stream Stats after {i+1} orders ===")
        print(processor.stats())


=== Stream Stats after 50 orders ===
{'total_orders': 50, 'total_revenue': 15194.829999999998, 'unique_customers': 50, 'window_avg_order_size': 303.8966, 'popular_product_last50': 'Monitor'}

=== Stream Stats after 100 orders ===
{'total_orders': 100, 'total_revenue': 28656.770000000004, 'unique_customers': 98, 'window_avg_order_size': 269.2388, 'popular_product_last50': 'Keyboard'}

=== Stream Stats after 150 orders ===
{'total_orders': 150, 'total_revenue': 48051.03, 'unique_customers': 141, 'window_avg_order_size': 387.88520000000005, 'popular_product_last50': 'Mouse'}


After processing all records through both batch and stream approaches, we compare the final cumulative statistics:

In [18]:
#step3
print("\n=== Final Comparison ===")
print("Batch total orders:", df.shape[0])
print("Batch total revenue:", df["total_amount"].sum())
print("Stream total orders:", processor.total_orders)
print("Stream total revenue:", processor.total_revenue)


=== Final Comparison ===
Batch total orders: 185
Batch total revenue: 59546.46
Stream total orders: 185
Stream total revenue: 59546.46


### Step 3 — Compare Final Results

After running both **batch** and **stream** processing, we compare the final totals:

- **Total orders** should be the same in both approaches.  
- **Total revenue** should also match.

**Why windowed streaming metrics can differ:**

- Stream processing often looks at only the most recent subset of orders (for example, the last 50 orders).  
- Batch processing uses all orders for the day or dataset.  
- Because of this, averages or most popular products in streaming can differ from daily batch results.

**What each approach tells us:**

- **Batch processing:** Shows complete, accurate daily or historical summaries. Good for reporting and analytics.  
- **Stream processing:** Shows real-time trends and patterns. Useful for monitoring and reacting quickly to changes.

**Conclusion:**

- Cumulative totals should match.  
- Streaming gives real-time insights, batch gives full context.
  

#### Task 5: Combine Batch and Stream Features
**Step 1**: Define a set of batch features and streaming features for a customer in an e-commerce system:

Batch features (computed from all historical data):

total_lifetime_orders
avg_order_amount
days_since_first_order
most_purchased_category
Streaming features (computed from last 10 orders):

recent_order_count
recent_avg_amount
seconds_since_last_order
recent_top_category

**Step 2**: For 5 sample customers, compute both batch and streaming features from the Task 1 data.

**Step 3**: Combine them into a single feature dictionary per customer. Print the combined feature vectors for all 5 customers.

**Step 4**: Write a markdown cell explaining why an ML model would benefit from having both batch and stream features. Give an example of a prediction task where the batch features alone would be insufficient, and another where the stream features alone would be insufficient

In [19]:
#step1
batch_features = ["total_lifetime_orders", "avg_order_amount", "days_since_first_order", "most_purchased_category"]
stream_features = ["recent_order_count", "recent_avg_amount", "seconds_since_last_order", "recent_top_category"]

In [20]:
import pandas as pd
from collections import Counter
from datetime import datetime

df = df_clean  

def compute_features(df, customer_id):
    cust_orders = df[df["customer_id"] == customer_id].sort_values("order_date")
    
    total_lifetime_orders = len(cust_orders)
    avg_order_amount = cust_orders["total_amount"].mean() if total_lifetime_orders > 0 else 0
    days_since_first_order = (datetime.now() - cust_orders["order_date"].min()).days if total_lifetime_orders > 0 else None
    most_purchased_category = cust_orders["amount_category"].mode()[0] if total_lifetime_orders > 0 else None
    
    last_10 = cust_orders.tail(10)
    recent_order_count = len(last_10)
    recent_avg_amount = last_10["total_amount"].mean() if recent_order_count > 0 else 0
    seconds_since_last_order = (datetime.now() - last_10["order_date"].max()).total_seconds() if recent_order_count > 0 else None
    recent_top_category = last_10["amount_category"].mode()[0] if recent_order_count > 0 else None
    
    features = {
        "customer_id": customer_id,
        # batch
        "total_lifetime_orders": total_lifetime_orders,
        "avg_order_amount": avg_order_amount,
        "days_since_first_order": days_since_first_order,
        "most_purchased_category": most_purchased_category,
        # stream
        "recent_order_count": recent_order_count,
        "recent_avg_amount": recent_avg_amount,
        "seconds_since_last_order": seconds_since_last_order,
        "recent_top_category": recent_top_category
    }
    
    return features

sample_customers = df["customer_id"].drop_duplicates().head(5)
all_features = [compute_features(df, cid) for cid in sample_customers]

In [21]:
#step3
for f in all_features:
    print(f)

{'customer_id': 1009, 'total_lifetime_orders': 1, 'avg_order_amount': np.float64(394.83000000000004), 'days_since_first_order': 421, 'most_purchased_category': 'large', 'recent_order_count': 1, 'recent_avg_amount': np.float64(394.83000000000004), 'seconds_since_last_order': 36394026.508355, 'recent_top_category': 'large'}
{'customer_id': 1822, 'total_lifetime_orders': 1, 'avg_order_amount': np.float64(514.44), 'days_since_first_order': 381, 'most_purchased_category': 'large', 'recent_order_count': 1, 'recent_avg_amount': np.float64(514.44), 'seconds_since_last_order': 32938026.514392, 'recent_top_category': 'large'}
{'customer_id': 1003, 'total_lifetime_orders': 1, 'avg_order_amount': np.float64(16.76), 'days_since_first_order': 382, 'most_purchased_category': 'small', 'recent_order_count': 1, 'recent_avg_amount': np.float64(16.76), 'seconds_since_last_order': 33024426.520274, 'recent_top_category': 'small'}
{'customer_id': 1301, 'total_lifetime_orders': 1, 'avg_order_amount': np.float

#step4
### Step 4 — Why Combine Batch and Stream Features

A machine learning model benefits from having both batch and stream features because they capture different aspects of customer behavior.

- **Batch features** summarize long-term behavior, like lifetime orders or average spending.  
- **Stream features** capture recent activity, like the last few orders, showing short-term trends.

**Example where batch features alone are insufficient:**  
- Predicting if a customer will participate in a flash sale. Lifetime behavior may not show sudden spikes in interest.

**Example where stream features alone are insufficient:**  
- Predicting customer lifetime value. Recent orders alone do not reflect the full purchase history.

**Conclusion:** Combining both types of features allows the model to use long-term patterns and short-term signals together for better predictions.